In [1]:
# !pip install -U langchain
# !pip install langchain langchain-community ollama
# !pip install -U langchain langchain-community langchain-core

In [2]:
# pip show langchain

Build a basic agent

In [3]:
from langchain_community.chat_models import ChatOllama

# Your tool
def get_weather(city: str) -> str:
    return f"It's always sunny in {city}!"

# Model
llm = ChatOllama(model="llama3:latest")

# Simple manual agent logic
def agent(user_input: str):
    if "weather" in user_input.lower():
        # extract city (very basic)
        city = user_input.split()[-1]
        return get_weather(city)
    
    return llm.invoke(user_input)

# Run
print(agent("what is the weather in sf"))

It's always sunny in sf!


C:\Users\USER\AppData\Local\Temp\ipykernel_6668\4002277855.py:8: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm = ChatOllama(model="llama3:latest")


In [4]:
from langchain_community.chat_models import ChatOllama

llm = ChatOllama(model="llama3:latest")

def get_weather(city: str) -> str:
    return f"It's always sunny in {city}!"

def agent(user_input: str):
    prompt = f"""
You are an assistant.

If the user asks about weather, respond EXACTLY like:
TOOL: get_weather(city)

Otherwise respond normally.

User: {user_input}
"""

    response = llm.invoke(prompt).content

    if "TOOL:" in response:
        city = response.split("(")[-1].replace(")", "")
        return get_weather(city)

    return response

print(agent("what is the weather in sf"))

It's always sunny in sf!


In [5]:
from dataclasses import dataclass
from langchain_community.chat_models import ChatOllama

# ---- TOOLS ----

def get_weather_for_location(city: str) -> str:
    return f"It's always sunny in {city}! ☀️"


@dataclass
class Context:
    user_id: str


def get_user_location(context: Context) -> str:
    user_id = context.user_id
    return "Florida" if user_id == "1" else "SF"


# ---- MODEL ----
llm = ChatOllama(model="llama3:latest")


# ---- SYSTEM PROMPT ----
SYSTEM_PROMPT = """You are an expert weather forecaster, who speaks in puns.

You have access to two tools:
- get_weather_for_location(city)
- get_user_location()

If you need a tool, respond EXACTLY like:
TOOL: tool_name(argument)

Examples:
TOOL: get_user_location()
TOOL: get_weather_for_location(london)
"""


# ---- AGENT LOOP ----
def agent(user_input: str, context: Context):
    prompt = SYSTEM_PROMPT + f"\nUser: {user_input}"

    response = llm.invoke(prompt).content
    print("LLM:", response)

    # ---- TOOL HANDLING ----
    if "TOOL:" in response:
        tool_call = response.split("TOOL:")[1].strip()

        # ---- TOOL 1 ----
        if "get_user_location" in tool_call:
            location = get_user_location(context)
            return agent(f"My location is {location}. What's the weather?", context)

        # ---- TOOL 2 ----
        elif "get_weather_for_location" in tool_call:
            city = tool_call.split("(")[-1].replace(")", "").strip()
            return get_weather_for_location(city)

    return response


# ---- RUN ----
ctx = Context(user_id="1")

print(agent("what's the weather like?", ctx))

LLM: I'm "fair" to say that I can help with that! TOOL: get_user_location()
LLM: A sunny disposition in Florida, eh?

TOOL: get_weather_for_location(Florida)
It's always sunny in Florida! ☀️


In [6]:
from langchain_community.chat_models import ChatOllama

model = ChatOllama(
    model="llama3:latest",
    temperature=0.5
)

response = model.invoke("Tell me a short joke about weather")
print(response.content)

Why did the meteorologist quit his job?

Because he couldn't forecast his future!


In [7]:
from dataclasses import dataclass
import json
from langchain_community.chat_models import ChatOllama

# ---- Schema ----
@dataclass
class ResponseFormat:
    punny_response: str
    weather_conditions: str | None = None


# ---- Model ----
llm = ChatOllama(model="llama3:latest")


# ---- Prompt ----
PROMPT = """
You are an expert weather forecaster who speaks in puns.

Respond ONLY in valid JSON format like this:
{{
  "punny_response": "your pun here",
  "weather_conditions": "optional weather info"
}}

User: {input}
"""


# ---- Function ----
def get_response(user_input: str) -> ResponseFormat:
    response = llm.invoke(PROMPT.format(input=user_input)).content
    print("RAW:", response)

    try:
        data = json.loads(response)
        return ResponseFormat(**data)
    except Exception:
        # fallback if model messes up JSON
        return ResponseFormat(
            punny_response=response,
            weather_conditions=None
        )


# ---- Run ----
result = get_response("What's the weather in SF?")
print(result)

RAW: {
  "punny_response": "San Francisco's forecast is 'groggy' with fog, but it'll 'shine' up later!",
  "weather_conditions": "Partly cloudy with a high of 58°F and a low of 48°F. Foggy mornings will give way to partly sunny skies in the afternoon."
}
ResponseFormat(punny_response="San Francisco's forecast is 'groggy' with fog, but it'll 'shine' up later!", weather_conditions='Partly cloudy with a high of 58°F and a low of 48°F. Foggy mornings will give way to partly sunny skies in the afternoon.')


Add memory

In [8]:
# !pip install langgraph

In [9]:
from typing import TypedDict, List
from langgraph.graph import StateGraph
from langgraph.checkpoint.memory import InMemorySaver
from langchain_community.chat_models import ChatOllama

# ---- 1. MODEL ----
llm = ChatOllama(model="llama3:latest")

# ---- 2. MEMORY ----
memory = InMemorySaver()

# ---- 3. STATE SCHEMA ----
class State(TypedDict):
    messages: List[dict]

# ---- 4. NODE FUNCTION ----
def chatbot(state: State):
    messages = state["messages"]

    # Call LLM
    response = llm.invoke(messages).content

    # Return NEW state (important!)
    return {
        "messages": messages + [
            {"role": "assistant", "content": response}
        ]
    }

# ---- 5. BUILD GRAPH ----
builder = StateGraph(State)

builder.add_node("chatbot", chatbot)
builder.set_entry_point("chatbot")
builder.set_finish_point("chatbot")

graph = builder.compile(checkpointer=memory)

# ---- 6. CONFIG (session id) ----
config = {"configurable": {"thread_id": "user1"}}

# ---- 7. RUN ----

# First message
result1 = graph.invoke(
    {"messages": [{"role": "user", "content": "Hello"}]},
    config
)
print("AI:", result1["messages"][-1]["content"])

# Second message (memory works here)
result2 = graph.invoke(
    {"messages": [{"role": "user", "content": "What did I say earlier?"}]},
    config
)
print("AI:", result2["messages"][-1]["content"])

AI: Hello! It's nice to meet you. Is there something I can help you with, or would you like to chat?
AI: This conversation just started, so you haven't said anything yet! I'm happy to chat with you, though. What's on your mind?


overview

In [12]:
from dataclasses import dataclass
from typing import Optional

from langchain_community.chat_models import ChatOllama

# ---- SYSTEM PROMPT ----
SYSTEM_PROMPT = """You are an expert weather forecaster, who speaks in puns.

You have access to tools:

- get_weather_for_location(city)
- get_user_location()

If user asks weather:
1. Find location
2. Then answer

Respond in JSON:
{
  "punny_response": "...",
  "weather_conditions": "..."
}
"""

# ---- CONTEXT ----
@dataclass
class Context:
    user_id: str

# ---- RESPONSE FORMAT ----
@dataclass
class ResponseFormat:
    punny_response: str
    weather_conditions: Optional[str] = None

# ---- TOOLS ----
def get_weather_for_location(city: str) -> str:
    return f"It's always sunny in {city}!"

def get_user_location(context: Context) -> str:
    return "Florida" if context.user_id == "1" else "SF"

# ---- MODEL (OLLAMA QWEN) ----
model = ChatOllama(
    model="qwen3.5",
    temperature=0,
    num_predict=200
)

# ---- SIMPLE MEMORY ----
memory = {}

# ---- AGENT (MANUAL REPLACEMENT) ----
def agent_invoke(messages, context: Context, thread_id="1"):
    state = memory.get(thread_id, [])

    # add system once
    if not state:
        state.append({"role": "system", "content": SYSTEM_PROMPT})

    state.extend(messages)

    # ---- STEP 1: get location ----
    if "weather" in messages[-1]["content"].lower():
        location = get_user_location(context)
        weather = get_weather_for_location(location)

        prompt = f"""
User asked weather.

Location: {location}
Weather: {weather}

Respond in JSON with pun.
"""

        response = model.invoke(prompt).content

        result = ResponseFormat(
            punny_response=response,
            weather_conditions=weather
        )

        state.append({"role": "assistant", "content": response})
        memory[thread_id] = state

        return {"structured_response": result}

    # ---- NORMAL RESPONSE ----
    response = model.invoke(messages[-1]["content"]).content

    result = ResponseFormat(
        punny_response=response,
        weather_conditions=None
    )

    state.append({"role": "assistant", "content": response})
    memory[thread_id] = state

    return {"structured_response": result}


# ---- RUN ----
config = {"configurable": {"thread_id": "1"}}

response = agent_invoke(
    [{"role": "user", "content": "what is the weather outside?"}],
    context=Context(user_id="1"),
    thread_id="1"
)

print(response["structured_response"])


response = agent_invoke(
    [{"role": "user", "content": "thank you!"}],
    context=Context(user_id="1"),
    thread_id="1"
)

print(response["structured_response"])

ResponseFormat(punny_response='', weather_conditions="It's always sunny in Florida!")
ResponseFormat(punny_response="You're very welcome! 😊 If you need anything else—whether it's help with a task, a question, or just a chat—feel free to ask anytime. Have a wonderful day! 🌟", weather_conditions=None)
